### Import Libraries

In [1]:
import apoc
from apoc import PixelClassifier
import os
import napari
from skimage.io import imread, imsave
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
import numpy as np
import shutil
import tifffile as tiff

### Select image

The path directories are adjusted to the file structure and filenames when acquiring images on the cellSens software from Evident in the "tiles" mode. Thus, images are organized in "Cycles". The original file structure was as follows:

experiment_number -> cycle_folder  -> frame_folder -> frames

In each frame folder a gray scale image of each channel and a mask from the machine learning step need to exist

In [ ]:
experiment= "5"
probe= "NR7_1nM_minus_benzo_tr1_"
frame_no = "G006_0005"

base_path=r"YOUR_PATH"

for exp in os.listdir(base_path):
    if experiment in exp:
        exp_folder= os.path.join(base_path, exp)

        for sample in os.listdir(exp_folder):
            if probe in sample:
                cycle_folder=  os.path.join(exp_folder, sample)

                for frame in os.listdir(cycle_folder):
                    if frame.endswith(f"{frame_no}.frames"):
                        frame_path = os.path.join(cycle_folder, frame)
                        frame_name = frame.removesuffix(".frames") # define a frame name for later to make it easier to save images
                        print(frame)

                        dapi = lyso = endo = nr = None
                        for file in os.listdir(frame_path):
                            full_path = os.path.join(frame_path, file)
                            if "C001" in file: dapi = imread(full_path)
                            elif "C002" in file: lyso = imread(full_path)
                            elif "C003" in file: endo = imread(full_path)
                            elif "C004" in file: nr   = imread(full_path)
                        
                        if all(x is not None for x in [dapi, lyso, endo, nr]):
                            print(f"Loaded: {full_path}")


image_stack=np.stack([dapi, lyso, endo, nr], axis=0) # all channels are stacked to generate one file

print(image_stack.dtype)
print(image_stack.shape)

### Annotate image

There are multiple ways to use napari and open/save images and to use multiple channels as input for apoc. Full documentation on [napari](https://github.com/napari/napari) and [apoc- nanapri plugin](https://github.com/haesleinhuepf/napari-accelerated-pixel-and-object-classification) can be found on GitHub. Here, all channels were stacked. Now the stack can be saved as all single grayscale images inside a single file. It is important that the label images and stacked images have the exact same name.

In [ ]:
# Define paths
stacked_path=f"YOUR_PATH/Stacks/Stacks_Exp_{experiment}/"
label_path= f"YOUR_PATH/Labels/Labels_Exp_{experiment}/"

# save stacked image
tiff.imwrite(f"{stacked_path}/{frame_name}_stack.tif", image_stack)

# Open saved image
# this step is not really necessary since we already have a defined variable for our stack, 
# but can be useful to check if the image has been saved correctly
image_stack_tif=f"{stacked_path}/{frame_name}_stack.tif" 

Next, the image can be opened in napari. Napari creates one layer for each channel. Visual adjustments like gamma values or channel colors can be adjusted beforehand or in napari itself.

In [ ]:
image = tiff.imread(image_stack_tif)

viewer = napari.Viewer()
viewer.add_image(image[0], name="DAPI", colormap="bop blue", blending="additive", gamma=0.8)
viewer.add_image(image[1], name="Lysosome", colormap="green", blending="additive", gamma=0.5)
viewer.add_image(image[2], name="Early Endosome", colormap="red", blending="additive", gamma=0.6)
viewer.add_image(image[3], name="Nanorods", colormap="magenta", blending= "additive", gamma=0.6)
label_layer= viewer.add_labels(label_img, name="Labels")

Once napari is opened a new label layer can be opened and the image can be annotated with labels. The label layer can be saved in *file -> save selected layers*. The label image needs to have the same name as the stacked image so we can print the correct name and copy it:

In [9]:
print(label_path)
print(f"{frame_name}_stack.tif")

NR7_1nM_minus_benzo_tr3_A01_G006_0005_stack.tif


Now that the labeling is done the PixelClassifier can be trained